# Self-consistency and best-of-N: hands-on (optional)
**LLMs, Animated · Lesson RS.2**

You'll build a tiny simulation of test-time compute. There is no language model here: a "solver" is just a random answer generator that is right with probability $p$ and otherwise picks a wrong answer from some distribution. That is enough to see *why* majority voting and best-of-N work, and when they stop working.

Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab). Runs in well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb
rng = np.random.default_rng(0)
CORRECT = 7                                   # the right answer to our imaginary question
print("ready")

## 1 · A noisy solver
Each sample from a model (at temperature above 0) is one full chain of reasoning ending in a final answer. We only care about that final answer. Our solver returns `CORRECT` with probability $p$; otherwise it returns a wrong answer drawn from `wrong_values` with probabilities `wrong_probs`.

Two kinds of wrong answers will matter:
- **spread out:** many different wrong values, each fairly rare (slips in different places),
- **concentrated:** one wrong value that dominates (a shared misconception).

In [ ]:
def sample_answers(n, p, wrong_values, wrong_probs, rng, correct=CORRECT):
    """Return an integer array of n final answers from our noisy solver.
    Each answer is `correct` with probability p. Otherwise it is one of `wrong_values`,
    drawn with probabilities `wrong_probs` (which sum to 1)."""
    # TODO 1: draw n booleans that are True with probability p   (hint: rng.random(n) < p)
    # TODO 2: draw n wrong answers with rng.choice(wrong_values, size=n, p=wrong_probs)
    # TODO 3: use np.where to pick the correct answer or the wrong one for each sample
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
SPREAD = (np.array([5, 6, 8, 9, 12, 14, 21, 36]), np.full(8, 1 / 8))   # 8 wrong answers, equally likely
SINGLE = (np.array([12]), np.array([1.0]))                               # always the same wrong answer
a = sample_answers(20000, 0.6, *SPREAD, rng=np.random.default_rng(1))
assert a.shape == (20000,) and np.issubdtype(a.dtype, np.integer)
assert abs(np.mean(a == CORRECT) - 0.6) < 0.02, "about 60% should be correct"
assert set(np.unique(a)) <= {CORRECT, *SPREAD[0].tolist()}
b = sample_answers(5000, 0.3, *SINGLE, rng=np.random.default_rng(2))
assert set(np.unique(b)) == {CORRECT, 12}
print("first answers:", a[:12])
print("✓ exercise 1")

## 2 · Majority vote (self-consistency)
Self-consistency (Wang et al., 2022) returns the most common final answer. With spread-out wrong answers there can be ties, so break them at random.

In [ ]:
def majority_vote(answers, rng):
    """Return the most common value in `answers` (a 1-D array).
    If several values tie for the most votes, pick one of them at random with rng."""
    # TODO: np.unique(answers, return_counts=True) gives the values and their counts.
    #       Find all values whose count equals the maximum, then rng.choice among them.
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
r = np.random.default_rng(3)
assert majority_vote(np.array([7, 7, 9, 7, 5]), r) == 7          # the lesson's animation
assert majority_vote(np.array([12, 7, 12]), r) == 12
assert majority_vote(np.array([4]), r) == 4
ties = [majority_vote(np.array([1, 2, 1, 2]), r) for _ in range(400)]
assert set(ties) == {1, 2} and 0.35 < np.mean(np.array(ties) == 1) < 0.65, "ties should be broken at random"
print("✓ exercise 2")

## 3 · The exact binomial
With independent samples, the number of correct ones $X$ is binomial. For odd $N$, the correct answer is guaranteed to win if it holds a strict majority:

$$P\left(X > \tfrac{N}{2}\right) = \sum_{k > N/2} \binom{N}{k} p^k (1-p)^{N-k}$$

In [ ]:
def maj_exact(N, p):
    """Probability that the correct answer gets MORE than half of N independent votes,
    when each vote is correct with probability p:  sum over k > N/2 of C(N, k) p^k (1-p)^(N-k)."""
    # TODO: loop k from N // 2 + 1 to N and add comb(N, k) * p**k * (1 - p)**(N - k)
    raise NotImplementedError("Exercise 3")

**Check 3.** These are the lesson's worked example, faded example and table.

In [ ]:
assert np.isclose(maj_exact(1, 0.7), 0.7)
assert np.isclose(maj_exact(3, 0.7), 0.784)
assert np.isclose(maj_exact(5, 0.6), 0.68256)
assert np.isclose(maj_exact(9, 0.7), 0.9012, atol=1e-4)
assert np.isclose(maj_exact(5, 0.4), 0.31744)
assert np.isclose(maj_exact(41, 0.4), 0.0965, atol=1e-4)
assert np.isclose(maj_exact(7, 0.5), 0.5), "a fair coin stays a coin flip"
print("✓ exercise 3")

## 4 · Simulate the vote and compare
Estimate majority-vote accuracy by simulation, then compare it with the formula.

- With a **single** wrong answer, every vote is "right" or "the wrong one", so the simulation should match `maj_exact` closely.
- With **spread-out** wrong answers, the correct answer can win with fewer than half of the votes, so the simulation should come out **at or above** the formula.

In [ ]:
def vote_accuracy(N, p, wrong_values, wrong_probs, rng, trials=3000):
    """Monte Carlo estimate: the fraction of trials in which majority_vote over N samples
    returns CORRECT."""
    # TODO: repeat `trials` times: sample N answers, vote, check whether the winner == CORRECT.
    #       Return the fraction of wins (a float between 0 and 1).
    raise NotImplementedError("Exercise 4")

**Check 4**

In [ ]:
r = np.random.default_rng(4)
for N in [1, 5, 9]:
    sim = vote_accuracy(N, 0.6, *SINGLE, rng=r)
    assert abs(sim - maj_exact(N, 0.6)) < 0.03, (N, sim, maj_exact(N, 0.6))
spread15 = vote_accuracy(15, 0.4, *SPREAD, rng=r)
print("p = 0.4, spread-out wrong answers, N = 15:", round(spread15, 3), "  formula (a floor):", round(maj_exact(15, 0.4), 3))
assert spread15 > 0.6, "with scattered mistakes, voting helps even at p = 0.4"
assert vote_accuracy(15, 0.4, *SINGLE, rng=r) < 0.3, "with one dominant wrong answer, voting hurts at p = 0.4"

Ns = [1, 3, 5, 9, 15, 21, 31]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for axi, p in zip(ax, [0.6, 0.4]):
    axi.plot(Ns, [maj_exact(N, p) for N in Ns], "k--", label="exact strict majority")
    axi.plot(Ns, [vote_accuracy(N, p, *SINGLE, rng=r, trials=1500) for N in Ns], "o-", label="sim: one wrong answer")
    axi.plot(Ns, [vote_accuracy(N, p, *SPREAD, rng=r, trials=1500) for N in Ns], "s-", label="sim: spread-out wrong answers")
    axi.axhline(p, color="grey", lw=0.8); axi.set_title(f"per-sample accuracy p = {p}"); axi.set_xlabel("samples N")
ax[0].set_ylabel("majority-vote accuracy"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("✓ exercise 4")

## 5 · Best-of-N with an imperfect verifier
Best-of-N doesn't vote: a verifier scores each sample and we keep the top one. Our verifier gives a correct sample a score 1 higher on average, plus Gaussian noise of size `sigma`.

- `sigma` near 0: a perfect verifier. Best-of-N is right whenever **any** sample is right, i.e. pass@N $= 1 - (1-p)^N$.
- `sigma` huge: the scores are pure noise, so best-of-N is just one random sample: accuracy $\approx p$.

In [ ]:
def best_of_n_accuracy(N, p, sigma, rng, trials=20000):
    """Best-of-N with an imperfect verifier, vectorised over trials.
    Each of the N samples is correct with probability p. The verifier's score is
        score = 1.0 * is_correct + noise,   noise ~ Normal(0, sigma)
    and best-of-N returns the sample with the highest score.
    Return the fraction of trials in which the chosen sample is correct."""
    # TODO 1: is_correct = rng.random((trials, N)) < p
    # TODO 2: scores = is_correct + rng.normal(0, sigma, size=(trials, N))
    # TODO 3: pick = scores.argmax(axis=1); chosen = is_correct[np.arange(trials), pick]
    # TODO 4: return chosen.mean()
    raise NotImplementedError("Exercise 5")

**Check 5**

In [ ]:
r = np.random.default_rng(5)
assert abs(best_of_n_accuracy(8, 0.3, 1e-6, r) - (1 - 0.7**8)) < 0.01, "perfect verifier = pass@N"
assert abs(best_of_n_accuracy(8, 0.3, 1e3, r) - 0.3) < 0.015, "useless verifier = one random sample"
mid = best_of_n_accuracy(8, 0.3, 0.7, r)
assert 0.3 < mid < 1 - 0.7**8
print("p = 0.3, N = 8: noisy verifier (sigma = 0.7) gets", round(mid, 3), "vs pass@8 =", round(1 - 0.7**8, 3))

Ns = [1, 2, 4, 8, 16, 32, 64]
plt.figure(figsize=(6, 3.8))
plt.plot(Ns, [1 - 0.7**N for N in Ns], "k--", label="pass@N (perfect checker)")
for s in [0.3, 0.7, 1.5]:
    plt.plot(Ns, [best_of_n_accuracy(N, 0.3, s, r, trials=6000) for N in Ns], "o-", label=f"best-of-N, sigma = {s}")
plt.plot(Ns, [vote_accuracy(N | 1, 0.3, *SPREAD, rng=r, trials=800) for N in Ns], "s:", label="majority vote (odd N), spread errors")
plt.xscale("log", base=2); plt.xlabel("samples N"); plt.ylabel("accuracy"); plt.legend(fontsize=8); plt.title("p = 0.3")
plt.tight_layout(); plt.show()
print("✓ exercise 5")

## Experiment: correlated errors
Real samples from one model are **not** independent: they share blind spots. A simple way to model that: with probability `rho`, all $N$ samples copy **one** shared draw (they "think alike"); otherwise they are independent. Run the cell and compare the curves.

Then try:
1. Change `rho` to 0.2 and 0.8. Where does each curve level off? (Check the lesson's claim: about $\rho p + (1 - \rho)$ when $p > 0.5$.)
2. **Questions differ.** Replace the fixed `p` with a different $p$ per question, e.g. `p_q = rng.beta(2, 1.2)`, so some questions are below 0.5. Average the vote accuracy over 300 questions. Why does the average curve flatten?
3. In exercise 5, what happens when the verifier is *systematically* fooled by one wrong answer? (Lesson RS.4 builds a playground on exactly this.)

In [ ]:
def vote_accuracy_correlated(N, p, rho, wrong_values, wrong_probs, rng, trials=2000):
    wins = 0
    for _ in range(trials):
        if rng.random() < rho:                                   # everyone thinks alike this time
            a = np.repeat(sample_answers(1, p, wrong_values, wrong_probs, rng), N)
        else:
            a = sample_answers(N, p, wrong_values, wrong_probs, rng)
        wins += majority_vote(a, rng) == CORRECT
    return wins / trials

r = np.random.default_rng(6)
p, Ns = 0.7, [1, 3, 5, 9, 15, 21, 31]
plt.figure(figsize=(6, 3.8))
for rho in [0.0, 0.4]:
    plt.plot(Ns, [vote_accuracy_correlated(N, p, rho, *SINGLE, rng=r) for N in Ns], "o-", label=f"rho = {rho}")
plt.axhline(0.4 * p + 0.6, color="grey", ls=":", label="0.4·p + 0.6")
plt.xlabel("samples N"); plt.ylabel("majority-vote accuracy"); plt.title(f"p = {p}"); plt.legend(); plt.tight_layout(); plt.show()

## Recap
- Majority voting needs comparable final answers and **diverse** samples; best-of-N needs a **scorer**.
- With independent samples, the binomial gives the chance of a strict majority. It rises with $N$ when $p > 0.5$ and falls when $p < 0.5$ with one dominant wrong answer.
- Spread-out mistakes help voting; **correlated** mistakes make it plateau.
- A perfect verifier turns best-of-N into pass@N; a noisy one lands in between.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def sample_answers(n, p, wrong_values, wrong_probs, rng, correct=CORRECT):
    right = rng.random(n) < p
    wrong = rng.choice(wrong_values, size=n, p=wrong_probs)
    return np.where(right, correct, wrong)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def majority_vote(answers, rng):
    values, counts = np.unique(answers, return_counts=True)
    winners = values[counts == counts.max()]
    return winners[0] if len(winners) == 1 else rng.choice(winners)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def maj_exact(N, p):
    return sum(comb(N, k) * p**k * (1 - p)**(N - k) for k in range(N // 2 + 1, N + 1))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def vote_accuracy(N, p, wrong_values, wrong_probs, rng, trials=3000):
    wins = 0
    for _ in range(trials):
        a = sample_answers(N, p, wrong_values, wrong_probs, rng)
        wins += majority_vote(a, rng) == CORRECT
    return wins / trials

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def best_of_n_accuracy(N, p, sigma, rng, trials=20000):
    is_correct = rng.random((trials, N)) < p
    scores = is_correct + rng.normal(0, sigma, size=(trials, N))
    pick = scores.argmax(axis=1)
    return is_correct[np.arange(trials), pick].mean()